# 04 Gold — card meta metrics

The report layer. Aggregates the silver card-grain table and quality check
table into small, dashboard-ready Delta tables:

- **`gold_card_metrics`** — one row per card: pick rate, win rate, sample size,
  enriched with `elixir_band` so the dashboard can cut by cost tier.
- **`gold_overview`** — a single KPI row for the dashboard's scalar tiles
  (totals, freshness, validity).
- **`gold_mode_breakdown`** — battles per game mode for the report day.

**Time scope:** This is a *daily* report. Bronze and silver keep the full
accumulated history; gold is scoped to the latest *complete* `battle_date`
(see the `report_date` below).

**Meta scope:** Trophy ladder only (`is_ladder = true`) for the card metrics.
Path of Legend and the other modes are deliberately excluded so win/pick rates
describe one metagame.

In [ ]:
from pyspark.sql import functions as F, types as T

CATALOG, SCHEMA = "workspace", "clash"

# Sources
SILVER_BATTLES     = f"{CATALOG}.{SCHEMA}.silver_battles"
SILVER_DECK_CARDS  = f"{CATALOG}.{SCHEMA}.silver_deck_cards"
DQ_RESULTS         = f"{CATALOG}.{SCHEMA}.dq_results"
QUARANTINE_BATTLES = f"{CATALOG}.{SCHEMA}.quarantine_battles"

# Targets (gold)
GOLD_CARD_METRICS = f"{CATALOG}.{SCHEMA}.gold_card_metrics"
GOLD_OVERVIEW     = f"{CATALOG}.{SCHEMA}.gold_overview"
GOLD_MODE_BREAKDOWN = f"{CATALOG}.{SCHEMA}.gold_mode_breakdown"

SCOPE_LABEL = "trophy_ladder"   # is_ladder = true
print("scope:", SCOPE_LABEL)

# Report on the latest *complete* day before today (UTC), using the partitioned silver table.
report_date = (spark.table(SILVER_BATTLES)
    .filter(F.col("battle_date") < F.current_date())
    .agg(F.max("battle_date")).first()[0])
if report_date is None:
    raise ValueError("No complete day of data in silver_battles (battle_date < today).")
print("report_date:", report_date)

## `gold_card_metrics`

`silver_deck_cards` is already at card grain: one row per card, per side, per
battle. For this layer we scope it to trophy ladder mode only, but all the
modes are kept in the database for other extended analysis:

- **pick rate** = appearances of a card / total deck instances (each battle has
  two decks, so the denominator is `2 × ladder battles`, minus any missing decks).
- **win rate** = that card's side wins / its appearances. `won` is the side's
  outcome from crown counts; equal-crown draws (~0.1%) count as non-wins.

`n_appearances` is kept as a plain column so the dashboard can filter out
tiny-sample cards instead of needing a confidence interval.

In [ ]:
# Card-grain rows, scoped to trophy ladder on the report day.
ladder_cards = (spark.table(SILVER_DECK_CARDS)
    .filter("is_ladder = true")
    .filter(F.col("battle_date") == F.lit(report_date)))

# Pick-rate denominator: distinct deck instances (battle_id x side) in scope.
total_decks = ladder_cards.select("battle_id", "side").distinct().count()
print("deck instances in scope:", total_decks)

gold_card = (ladder_cards
    .groupBy("card_id", "card_name", "rarity", "elixir_cost", "elixir_band")
    .agg(
        F.count("*").alias("n_appearances"),
        # count(won) ignores nulls, so the win-rate denominator only counts
        # battles whose outcome was decidable (crowns present on both sides).
        F.count("won").alias("n_decided"),
        F.sum(F.col("won").cast("int")).alias("wins"),
    )
    .withColumn("pick_rate", F.col("n_appearances") / F.lit(total_decks))
    .withColumn("win_rate", F.col("wins") / F.col("n_decided"))
    .withColumn("scope", F.lit(SCOPE_LABEL))
    .withColumn("report_date", F.lit(report_date))
    .withColumn("generated_at", F.current_timestamp()))

(gold_card.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(GOLD_CARD_METRICS))

gc = spark.table(GOLD_CARD_METRICS)
print(gc.count(), "cards in gold_card_metrics")
display(gc.orderBy(F.desc("pick_rate")).limit(10))

## `gold_overview`

One KPI row for the dashboard's scalar tiles, scoped to the report day. Counts
come from silver (filtered to `report_date`); `freshness_hours` is read from the
latest `dq_results` run since recency is a pipeline-wide signal, not a per-day one.

- `distinct_participants` — every tag seen on either side that day. Mostly one-off
  opponents we never crawled.
- `validity_pct` — share of the day's battles that passed every *error* check
  (i.e. were not quarantined). The quarantine count is filtered to `report_date`
  so it lines up with `total_battles`.

In [ ]:
# Battle counts scoped to the report day.
sb = spark.table(SILVER_BATTLES).filter(F.col("battle_date") == F.lit(report_date))
total_battles  = sb.count()
ladder_battles = sb.filter("is_ladder = true").count()

# Distinct participants: union of both sides\' tags (dominated by one-off opponents).
participants = (sb.select(F.col("player_tag").alias("tag"))
    .union(sb.select(F.col("opponent_tag").alias("tag")))
    .filter(F.col("tag").isNotNull()).distinct().count())

distinct_cards = spark.table(GOLD_CARD_METRICS).count()

# Freshness from the latest dq_results run (recency is pipeline-wide, not day-scoped).
latest_run = spark.table(DQ_RESULTS).agg(F.max("run_id")).first()[0]
dq = spark.table(DQ_RESULTS).filter(F.col("run_id") == latest_run)
freshness_hours = (dq.filter("check_name = 'freshness_under_24h'")
    .agg(F.max("metric_value")).first()[0])

# Validity scoped to the report day so the ratio matches total_battles above.
quarantined = (spark.table(QUARANTINE_BATTLES)
    .filter(F.col("battle_date") == F.lit(report_date))
    .select("battle_id").distinct().count())
validity_pct = 100.0 * (1 - quarantined / total_battles) if total_battles else None

overview_schema = T.StructType([
    T.StructField("report_date", T.DateType()),
    T.StructField("total_battles", T.LongType()),
    T.StructField("ladder_battles", T.LongType()),
    T.StructField("distinct_participants", T.LongType()),
    T.StructField("distinct_cards", T.LongType()),
    T.StructField("freshness_hours", T.DoubleType()),
    T.StructField("validity_pct", T.DoubleType()),
    T.StructField("scope", T.StringType()),
    T.StructField("dq_run_id", T.StringType()),
])
overview = spark.createDataFrame([(
    report_date, total_battles, ladder_battles, participants, distinct_cards,
    float(freshness_hours) if freshness_hours is not None else None,
    float(validity_pct) if validity_pct is not None else None,
    SCOPE_LABEL, latest_run,
)], overview_schema).withColumn("generated_at", F.current_timestamp())

(overview.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(GOLD_OVERVIEW))
display(spark.table(GOLD_OVERVIEW))

## `gold_mode_breakdown`

Battle count and share of the corpus per game mode for the report day,
across **all** modes (not scoped to ladder). This is what justifies the
trophy-ladder scope of the metric tables — it shows how big the `Ladder` slice
is relative to Path of Legend and everything else on the day. The `is_ladder`
flag lets the dashboard highlight the in-scope slice.

In [ ]:
# Mode breakdown scoped to the report day, across all modes.
sb = spark.table(SILVER_BATTLES).filter(F.col("battle_date") == F.lit(report_date))
mode_total = sb.count()

gold_mode = (sb
    .groupBy("game_mode", "is_ladder")
    .agg(F.count("*").alias("n_battles"))
    .withColumn("pct_of_total", 100.0 * F.col("n_battles") / F.lit(mode_total))
    .withColumn("report_date", F.lit(report_date))
    .withColumn("generated_at", F.current_timestamp())
    .orderBy(F.desc("n_battles")))

(gold_mode.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(GOLD_MODE_BREAKDOWN))

print(f"{spark.table(GOLD_MODE_BREAKDOWN).count()} game modes; "
      f"ladder share = "
      f"{spark.table(GOLD_MODE_BREAKDOWN).filter('is_ladder').agg(F.sum('pct_of_total')).first()[0]:.1f}%")
display(spark.table(GOLD_MODE_BREAKDOWN).limit(20))

## Dashboard preview — the two headline charts

Top cards by win rate (with a minimum-sample floor so a rarely-played card can't
spike the chart) and top cards by pick rate.

In [ ]:
gc = spark.table(GOLD_CARD_METRICS)

MIN_APPEARANCES = 30   # floor out tiny-sample cards from the win-rate chart
print(f"top cards by win rate (>= {MIN_APPEARANCES} appearances)")
display(gc.filter(F.col("n_appearances") >= MIN_APPEARANCES)
    .orderBy(F.desc("win_rate"))
    .select("card_name", "rarity", "elixir_cost", "win_rate", "pick_rate", "n_appearances")
    .limit(10))

print("top cards by pick rate")
display(gc.orderBy(F.desc("pick_rate"))
    .select("card_name", "rarity", "elixir_cost", "pick_rate", "win_rate", "n_appearances")
    .limit(10))